# HepatoNet reduction with fastCORE for PhysiCell-dFBA

This notebook documents the reduction workflow used to generate and validate the fastCORE-based HepatoNet candidate networks for the hepatic lobule model.

The workflow keeps the original project files separated from the reduced-network outputs and uses the previously defined zone-specific constraints for zones 1/2 and zone 3.


## Main outputs

Validated SBML files generated by the workflow:

- `notebooks/fastcore_outputs/official_fastcore/validated/zone12_official_fastcore_literature_validated_reduced.xml`
- `notebooks/fastcore_outputs/official_fastcore/validated/zone3_official_fastcore_literature_validated_reduced.xml`

Copies used for PhysiCell coupling:

- `config/hepatonet_fastcore_validated.xml`
- `config/hepatonet_zone3_fastcore_validated.xml`


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
NOTEBOOKS = ROOT / 'notebooks'
OUT = NOTEBOOKS / 'fastcore_outputs'
FASTCORE_OUT = OUT / 'official_fastcore'
VALIDATED_OUT = FASTCORE_OUT / 'validated'

sys.path.insert(0, str(NOTEBOOKS))

print('Project root:', ROOT)
print('fastCORE output:', FASTCORE_OUT)


## 1. Input cores

These CSV files are the literature-guided core reaction sets. They include carbohydrate metabolism, O2/CO2 exchange, lactate-related reactions, glutamine/glutamate, amino acid metabolism and nitrogen-related reactions.


In [ ]:
core_files = {
    'zone12_literature_guided': OUT / 'zone12_literature_guided_core.csv',
    'zone3_literature_guided': OUT / 'zone3' / 'zone3_literature_guided_core.csv',
    'zone12_compact_functional': OUT / 'zone12_compact_functional_core.csv',
    'zone3_compact_functional': OUT / 'zone3' / 'zone3_compact_functional_core.csv',
}

for name, path in core_files.items():
    df = pd.read_csv(path)
    print(f'{name}: {len(df)} reactions -> {path}')


## 2. Load the reduction code

The main implementation is in `notebooks/run_official_fastcore_literature.py`. It contains:

- zone-specific HepatoNet preparation/bounds;
- a COBRApy/GLPK implementation following the fastCORE logic;
- SBML export;
- validation tests for `r1032`, `r1389`, O2/CO2, lactate and PhysiCell exchange IDs.


In [ ]:
import run_official_fastcore_literature as fc

print('HepatoNet zone 1/2:', fc.HEPATONET)
print('HepatoNet zone 3:', fc.HEPATONET_ZONE3)
print('Output folder:', fc.FASTCORE_OUT)


## 3. Optional: run fastCORE reduction

This cell reruns the full reduction. It can take several minutes because the literature-guided cores have more than 1,300 reactions per zonal model.

If the XMLs already exist, you can skip this cell and go directly to validation.


In [ ]:
# Long-running cell: uncomment to regenerate the raw fastCORE networks.
# fc.main()


## 4. Post-process fastCORE networks for PhysiCell validation

Raw fastCORE can remove a closed exchange such as lactate if it is not required in the basal solution. For PhysiCell-dFBA coupling and reviewer-facing tests, we preserve the compact functional core on top of the fastCORE result.

This generates the `*_validated_reduced.xml` files.


In [ ]:
import postprocess_fastcore_validated as post

# Uncomment to regenerate validated SBMLs after rerunning fastCORE.
# post.main()


## 5. Inspect final validated networks


In [ ]:
summary_path = VALIDATED_OUT / 'official_fastcore_literature_validated_summary.csv'
summary = pd.read_csv(summary_path)
summary


## 6. Validation tests

The final networks are tested for:

- zone 1/2 objective: `r1032`;
- zone 3 objective: `r1389`;
- simplified oxidation: `Glc + 6O2 -> 6CO2 + 6H2O`;
- simplified lactate route: `Glc -> 2 lactate`;
- exchange IDs used by `PhysiCell_settings.xml`.


In [ ]:
validation_path = VALIDATED_OUT / 'official_fastcore_literature_validated_validation_report.csv'
validation = pd.read_csv(validation_path)
validation


In [ ]:
validation.pivot_table(
    index='network',
    columns='test',
    values='status',
    aggfunc='first'
)


## 7. Copy validated SBMLs to `config/` for PhysiCell

This cell copies the validated reduced networks to the configuration folder. It does not run the PhysiCell simulation.


In [ ]:
from shutil import copy2

src_zone12 = VALIDATED_OUT / 'zone12_official_fastcore_literature_validated_reduced.xml'
src_zone3 = VALIDATED_OUT / 'zone3_official_fastcore_literature_validated_reduced.xml'
dst_zone12 = ROOT / 'config' / 'hepatonet_fastcore_validated.xml'
dst_zone3 = ROOT / 'config' / 'hepatonet_zone3_fastcore_validated.xml'

# Uncomment if you want to copy/update the SBML files in config/.
# copy2(src_zone12, dst_zone12)
# copy2(src_zone3, dst_zone3)

print(dst_zone12)
print(dst_zone3)


## 8. Static PhysiCell coupling check

This check confirms that the SBML files referenced by `PhysiCell_settings.xml` contain the required objective reactions and exchange reaction IDs.


In [ ]:
# This script writes:
# notebooks/fastcore_outputs/official_fastcore/validated/physicell_fastcore_static_coupling_check.csv

%run notebooks/check_fastcore_physicell_coupling.py


## Notes for reporting

The final validated fastCORE networks should be described as candidate reduced networks. They preserve the modules requested by the reviewers, including carbohydrate metabolism, O2/CO2 exchange, lactate testing capacity, and nitrogen/amino-acid related reactions.

They should not yet be presented as a definitive replacement for full HepatoNet1 until longer coupled simulations and zone-specific calibration are completed.
